In [1]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path
import pickle
import json
import requests
import anndata
from anndata import AnnData
from typing import Dict

In [2]:
def pickle_data(data, path):

    with open(path, "wb") as f:
        pickle.dump(data, f)

def load_pickle(path): 

    with open(path, "rb") as f: 
        return pickle.load(f)

## RNA

Fix relative to the original version of this notebook: CPTAC has tumor **and** matched normal RNA-seq for most cases (2,393 files for ~1,300 cases). The original notebook loaded every file and `get_CPTAC_embeddings.py` then kept whichever file per case came last, so 240 of 1,222 cohort cases were represented by solid-tissue normal. Here each case keeps exactly one **Primary Tumor** file (lowest sample barcode, then file name), chosen before gene filtering and variance computation.

Sample types come from `clinical/rna_file_sample_types.csv`, a GDC `/files` lookup of every RNA file (re-queried if the file is missing).

In [3]:
cptac_folder = Path("/data/rbg/shared/datasets/CPTAC/CPTAC3_gene_counts")
SAMPLE_TYPES = Path("../../data/CPTAC/clinical/rna_file_sample_types.csv")
MIN_MEAN_TPM = 0.1

In [4]:
def query_sample_types(raw_dir):
    rows = [(d, f) for d in os.listdir(raw_dir) for f in os.listdir(raw_dir / d)
            if 'augmented_star_gene_counts.tsv' in f]
    df = pd.DataFrame(rows, columns=['dir', 'file_name'])
    info, names = {}, df['file_name'].tolist()
    for i in range(0, len(names), 500):
        r = requests.post('https://api.gdc.cancer.gov/files', timeout=180, json={
            'filters': {'op': 'in', 'content': {'field': 'file_name', 'value': names[i:i + 500]}},
            'fields': 'file_name,cases.case_id,cases.submitter_id,cases.project.project_id,'
                      'cases.samples.sample_type,cases.samples.submitter_id,'
                      'cases.samples.tumor_descriptor',
            'size': 500})
        for h in r.json()['data']['hits']:
            c, s = h['cases'][0], h['cases'][0]['samples'][0]
            info[h['file_name']] = (c['case_id'], c['submitter_id'], c['project']['project_id'],
                                    s['sample_type'], s['submitter_id'], s.get('tumor_descriptor'))
    cols = ['case_id', 'case_submitter', 'project', 'stype', 'sample_submitter', 'tumor_descriptor']
    df[cols] = pd.DataFrame([info.get(f, (None,) * 6) for f in df['file_name']], index=df.index)
    return df

if not SAMPLE_TYPES.exists():
    query_sample_types(cptac_folder).to_csv(SAMPLE_TYPES, index=False)
files = pd.read_csv(SAMPLE_TYPES)
print(f'RNA files: {len(files)}; cases: {files.case_id.nunique()}')
print(files.stype.value_counts())

RNA files: 2393; cases: 1342
stype
Primary Tumor          1693
Solid Tissue Normal     668
Recurrent Tumor          21
Metastatic               11
Name: count, dtype: int64


In [5]:
tumor = files[files.stype == 'Primary Tumor']
chosen = tumor.sort_values(['case_id', 'sample_submitter', 'file_name']).drop_duplicates('case_id')
print(f'Cases with a primary tumor file: {len(chosen)}; without (dropped): '
      f'{files.case_id.nunique() - len(chosen)}')

Cases with a primary tumor file: 1309; without (dropped): 33


In [6]:
# Same parsing as before: tpm_unstranded per gene_name (a later duplicate gene_name overwrites)
expr, gene_type = {}, {}
for _, r in tqdm(chosen.iterrows(), total=len(chosen), desc='Loading TPM files'):
    e = pd.read_csv(cptac_folder / r.dir / r.file_name, sep='\t', skiprows=1).fillna('-')
    e = e[e['gene_name'] != '-']
    expr[r.file_name] = dict(zip(e['gene_name'], e['tpm_unstranded']))
    if not gene_type:
        gene_type = dict(zip(e['gene_name'], e['gene_type']))
df = pd.DataFrame.from_dict(expr, orient='index').astype(np.float32)
print(f'Samples x genes: {df.shape}')

Loading TPM files:   0%|          | 0/1309 [00:00<?, ?it/s]

Loading TPM files:   0%|          | 1/1309 [00:00<05:37,  3.87it/s]

Loading TPM files:   0%|          | 2/1309 [00:00<04:40,  4.66it/s]

Loading TPM files:   0%|          | 3/1309 [00:00<04:27,  4.89it/s]

Loading TPM files:   0%|          | 4/1309 [00:00<04:26,  4.90it/s]

...

Loading TPM files: 100%|█████████▉| 1307/1309 [03:42<00:00,  6.29it/s]

Loading TPM files: 100%|█████████▉| 1308/1309 [03:42<00:00,  6.30it/s]

Loading TPM files: 100%|██████████| 1309/1309 [03:42<00:00,  6.31it/s]

Loading TPM files: 100%|██████████| 1309/1309 [03:42<00:00,  5.88it/s]

Samples x genes: (1309, 59427)


In [7]:
# Filter genes with mean TPM over 0.1, log1p, per-gene variance
df = df.loc[:, df.mean(axis=0) > MIN_MEAN_TPM]
X_log = np.log1p(df.values)
obs = pd.DataFrame({'patient_id': df.index}, index=df.index.astype(str))
obs['case_id'] = chosen.set_index('file_name').loc[df.index, 'case_id'].values
var = pd.DataFrame(index=df.columns.astype(str))
var['gene_symbol'] = var.index
var['log_variance'] = X_log.var(axis=0)
print(f'Genes with mean TPM > {MIN_MEAN_TPM}: {df.shape[1]}')

Genes with mean TPM > 0.1: 46783


In [8]:
os.makedirs('../../data/CPTAC/RNA', exist_ok=True)
filtered_adata = AnnData(X=df.values, obs=obs, var=var)
filtered_adata.write_h5ad('../../data/CPTAC/RNA/CPTAC_bulk_RNA_processed.h5ad')
log_adata = AnnData(X=X_log.astype(np.float32), obs=obs, var=var)
log_adata.write_h5ad('../../data/CPTAC/RNA/CPTAC_bulk_RNA_processed_log1p.h5ad')
filtered_adata

AnnData object with n_obs × n_vars = 1309 × 46783
    obs: 'patient_id', 'case_id'
    var: 'gene_symbol', 'log_variance'

In [9]:
# Protein-coding genes ordered by log1p variance (baseline input)
pc = [g for g in var.index if gene_type.get(g) == 'protein_coding']
selected_genes_cptac = dict(var.loc[pc, 'log_variance'].sort_values(ascending=False).items())
print(len(selected_genes_cptac))
pickle_data(selected_genes_cptac, '../../data/CPTAC/CPTAC_genes_ordered_by_variance.pkl')

18255


### Get cancer type mapping

In [10]:
RNA_PATH = "../../data/CPTAC/RNA/CPTAC_bulk_RNA_processed_log1p.h5ad"
CLINICAL_PATH = "../../data/CPTAC/clinical/clinical.tsv"
METADATA_JSON = "../../data/CPTAC/clinical/metadata.cohort.all.json"

In [11]:
adata = anndata.read_h5ad(RNA_PATH)
obs_ids = adata.obs_names.astype(str).tolist()

print(f"Loaded RNA AnnData with {len(obs_ids)} samples")

Loaded RNA AnnData with 1309 samples


In [12]:
with open(METADATA_JSON, "r") as f:
    metadata = json.load(f)

file_to_case = {}

for r in metadata:
    ae = r.get("associated_entities", [])
    if not ae:
        continue

    case_id = ae[0].get("case_id", None)
    if case_id is None:
        continue

    for da in r.get("downstream_analyses", []):
        for of in da.get("output_files", []):
            if of.get("data_type") == "Gene Expression Quantification":
                fn = of.get("file_name", None)
                if fn:
                    file_to_case[fn] = case_id

print(f"Mapped {len(file_to_case)} RNA files to case_ids")

Mapped 2760 RNA files to case_ids


In [13]:
# -----------------------------
# Load clinical and build case_id -> organ mapping
# -----------------------------
clinical = pd.read_csv(CLINICAL_PATH, sep="\t")
clinical["cases.case_id"] = clinical["cases.case_id"].astype(str)

organ_cols = [
    "diagnoses.tissue_or_organ_of_origin",
    "diagnoses.primary_site",
    "diagnoses.site_of_resection_or_biopsy",
    "diagnoses.primary_diagnosis",
]

case_to_organ = {}

for _, row in clinical.iterrows():
    case_id = row["cases.case_id"]
    organ = None
    for col in organ_cols:
        if col in row.index and pd.notna(row[col]):
            organ = str(row[col]).strip()
            break
    case_to_organ[case_id] = organ

print(f"Mapped {len(case_to_organ)} case_ids to organ")

Mapped 1345 case_ids to organ


In [14]:
# -----------------------------
# Final mapping: obs_name -> organ
# -----------------------------
obs_to_organ = {}

for obs in obs_ids:
    # remove trailing .pkl to match metadata file_name
    file_name = obs[:-4] if obs.endswith(".pkl") else obs
    case_id = file_to_case.get(file_name, None)

    if case_id is None:
        obs_to_organ[obs] = None
    else:
        obs_to_organ[obs] = case_to_organ.get(case_id, None)

print("Finished building obs_to_organ mapping")

# Example
list(obs_to_organ.items())[:5]

Finished building obs_to_organ mapping


[('f093ea86-510e-403e-8568-d0486f5ec824.rna_seq.augmented_star_gene_counts.tsv',
  'Kidney, NOS'),
 ('514aa2ae-6c49-4b89-9a44-0f87a80bf77f.rna_seq.augmented_star_gene_counts.tsv',
  'Lung, NOS'),
 ('2fc7f327-e498-4857-ab29-1849fe15c00d.rna_seq.augmented_star_gene_counts.tsv',
  'Kidney, NOS'),
 ('f9806a31-d0e2-4bea-858b-3934067ba2f4.rna_seq.augmented_star_gene_counts.tsv',
  'Lung, NOS'),
 ('648e135c-b905-4eb6-9a13-3ba65e4ad2af.rna_seq.augmented_star_gene_counts.tsv',
  'Brain, NOS')]

In [15]:
organ_counts = pd.Series(list(obs_to_organ.values())).value_counts()

In [16]:
organ_counts

Lung, NOS                                             337
Kidney, NOS                                           261
Uterus, NOS                                           240
Brain, NOS                                            178
Pancreas, NOS                                         160
Larynx, NOS                                            49
Tongue, NOS                                            21
Floor of mouth, NOS                                    18
Temporal lobe                                           9
Parietal lobe                                           5
Overlapping lesion of lip, oral cavity and pharynx      5
Lip, NOS                                                4
Frontal lobe                                            4
Oropharynx, NOS                                         4
Gum, NOS                                                3
Head, face or neck, NOS                                 2
Cheek mucosa                                            2
Unknown       

In [17]:
cptac_to_tcga = {
    "Lung, NOS": "LUAD",
    "Kidney, NOS": "KIRC",
    "Uterus, NOS": "UCEC",
    "Pancreas, NOS": "PAAD",
    "Brain, NOS": "GBM",
    "Breast, NOS": "BRCA",
    "Colon, NOS": "COAD",
}

In [18]:
# print number of samples that have TCGA mapping
mapped_samples = sum(1 for organ in obs_to_organ.values() if organ in cptac_to_tcga)
mapped_samples

1176

In [19]:
# Create obs_to_tcga mapping that builds off obs_to_organl this mapping should be a dict, keys being obs
obs_to_tcga = {}
for obs, organ in obs_to_organ.items():
    if organ in cptac_to_tcga:
        obs_to_tcga[obs] = cptac_to_tcga[organ]

print(f"Mapped {len(obs_to_tcga)} samples to TCGA subtypes")
pickle_data(obs_to_tcga, "../../data/CPTAC/obs_to_tcga_mapping.pkl")

Mapped 1176 samples to TCGA subtypes


### Case-level TCGA mapping (used to restrict COMPASS to mappable cases)

In [20]:
tcga_dict = load_pickle('../../data/CPTAC/obs_to_tcga_mapping.pkl')

In [21]:
def load_cptac_rna_filename_to_case_id(metadata_json_path: Path):
    """
    Builds mapping:
      "<file_name>" -> "<case_id>"
    where file_name looks like:
      ffeda85e-....rna_seq.augmented_star_gene_counts.tsv
    """
    with open(metadata_json_path, "r") as f:
        meta = json.load(f)

    file_to_case: Dict[str, str] = {}
    for r in meta:
        ae = r.get("associated_entities", [])
        if not ae:
            continue
        case_id = ae[0].get("case_id", None)
        if case_id is None:
            continue

        for da in r.get("downstream_analyses", []):
            for of in da.get("output_files", []):
                if of.get("data_type") == "Gene Expression Quantification":
                    fn = of.get("file_name", None)
                    if fn:
                        file_to_case[fn] = case_id

    return file_to_case

In [22]:
file_to_case = load_cptac_rna_filename_to_case_id(Path(METADATA_JSON))

In [23]:
id_tcga_mapping = {file_to_case[k]: v for k, v in tcga_dict.items() if k in file_to_case}

In [24]:
len(id_tcga_mapping), len(tcga_dict)

(1176, 1176)

In [25]:
pickle_data(id_tcga_mapping, '../../data/CPTAC/id_to_tcga_mapping.pkl')

### Analyze CPTAC embeddings

Runs only after `get_CPTAC_embeddings.py` has written `data/RNA_embeddings/CPTAC_processed.pkl`.

In [26]:
PKL = Path('../../data/RNA_embeddings/CPTAC_processed.pkl')
if PKL.exists():
    new_cptac = load_pickle(PKL)
    print('Total records:', len(new_cptac))
    for k in ['organ', 'cancer_type']:
        print(f'Missing {k}:', sum(rec.get(k) is None for rec in new_cptac.values()))
    print(pd.Series([rec.get('organ') for rec in new_cptac.values()]).value_counts())
else:
    print(f'{PKL} not built yet')

../../data/RNA_embeddings/CPTAC_processed.pkl not built yet
